# 第十四章 · 在CUDA上补数字

[![在Colab中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zxh0/vibe-writing/blob/main/toyllm_book1/code/book/ch14/colab.ipynb)

第十四章正文里有5处`TODO CUDA`，这个笔记本把它们要的输出一次跑完：

| 脚本 | 跑什么 | 补正文哪一节 |
|---|---|---|
| `check_device.py` | SmolLM2、Qwen3-0.6B-Base各一次 | 第一次核对、半精度 |
| `speed.py` | SmolLM2、Qwen3-0.6B-Base各一次 | 速度、半精度 |
| `batch.py` | SmolLM2，fp32和半精度各一次，B到64 | 批处理 |
| `flash.py` | fp32和半精度各一次，T到16384 | FlashAttention |

**用法**：点上面的按钮在Colab里打开，菜单“代码执行程序 → 更改运行时类型”选GPU（免费的是T4），
然后“全部运行”。代码从[vibe-writing](https://github.com/zxh0/vibe-writing/tree/main/toyllm_book1)仓库克隆，不用上传任何东西。所有输出存进`/content/results/`，最后打包成一个zip下载。

**T4要注意两件事**，第三个格子会实测出来：

1. T4是Turing架构（sm75），**没有原生BF16**。BF16能算、精度数字照样有意义，但速度没有参考价值，
   所以速度和批处理用FP16（下面的`HALF`会自动挑）。
2. PyTorch的FlashAttention内核要sm80及以上（A100、L4、A10……）。T4上`F.scaled_dot_product_attention()`
   挑的是memory-efficient attention，它同样不建那张T×T的表，峰值显存同样随T线性增长，但名字不叫FlashAttention。
   另外，**FlashAttention内核只收FP16/BF16**，FP32在哪张卡上都不会走它，所以`flash.py`两种精度都跑。

用的是Colab自带的torch，不装书里那个`torch<2.3`（那是作者的Intel Mac的限制）。第十四章的脚本只用
`torch`、`safetensors`、`tokenizers`，Colab都有。

In [ ]:
# ---- 配置 ----------------------------------------------------------------
REPO_URL = "https://github.com/zxh0/vibe-writing.git"
BRANCH = "main"
SUBDIR = "toyllm_book1/code"

RUN_QWEN = True          # 再下约1.5GB的Qwen3-0.6B-Base，跑check_device和speed
MAX_BATCH = 64
MAX_T = 16384

In [ ]:
import os, pathlib, shutil, subprocess, sys

def sh(cmd, cwd=None):
    """跑一条shell命令，边跑边打印，失败就停。"""
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

# 稀疏克隆：整个仓库有好几本书，这里只取第十四章的代码和下载权重的脚本。
WORK = pathlib.Path("/content/vibe-writing")
if not WORK.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} --filter=blob:none --no-checkout {REPO_URL} {WORK}")
    sh(f"git sparse-checkout set --no-cone /{SUBDIR}/book/ch14/ /{SUBDIR}/download_model.sh", cwd=WORK)
    sh("git checkout", cwd=WORK)
else:
    sh("git pull --ff-only", cwd=WORK)
sh("git log -1 --format='%h %ad %s' --date=short", cwd=WORK)

CODE = WORK / SUBDIR
os.chmod(CODE / "download_model.sh", 0o755)
# attend_sdpa要先补一维批次：三维的输入会悄悄退回math内核，峰值显存随T平方增长，
# flash.py量出来的就全是错的。仓库里还是旧版时，停在这里。
assert "q.dim() == 3" in (CODE / "book/ch14/model.py").read_text(),     "仓库里的ch14/model.py还没有attend_sdpa补批次那一维的修正，先同步再跑"
RESULTS = pathlib.Path("/content/results")
RESULTS.mkdir(exist_ok=True)
print("代码在", CODE)

## 环境

记下GPU型号、驱动、torch版本，再实测两件事：有没有原生BF16，以及`F.scaled_dot_product_attention()`
在这张卡上三种内核各能不能用。输入用四维的`(1, 9, T, 64)`，和`model.py`传进去的一样。

In [ ]:
import torch, torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

lines = []
def log(s=""):
    print(s)
    lines.append(s)

assert torch.cuda.is_available(), "没有CUDA：菜单“代码执行程序 → 更改运行时类型”选GPU"
p = torch.cuda.get_device_properties(0)
cap = torch.cuda.get_device_capability()
log(f"GPU      {p.name}，sm{cap[0]}{cap[1]}，显存{p.total_memory / 2**30:.1f}GiB")
log(f"torch    {torch.__version__}，CUDA {torch.version.cuda}，Python {sys.version.split()[0]}")
driver = subprocess.run("nvidia-smi --query-gpu=driver_version --format=csv,noheader",
                        shell=True, capture_output=True, text=True).stdout.strip()
log(f"驱动     {driver}，CPU线程{torch.get_num_threads()}个")
native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
log(f"原生BF16 {'有' if native_bf16 else '没有（能算，但是模拟的，慢）'}")
HALF = "bfloat16" if native_bf16 else "float16"
log(f"速度和批处理用 {HALF}")

log("\nF.scaled_dot_product_attention各内核能不能用，(1, 9, 1024, 64)，is_causal=True")
q = torch.randn(1, 9, 1024, 64, device="cuda")
for dt in (torch.float32, torch.float16, torch.bfloat16):
    row = f"  {str(dt).split('.')[-1]:<10}"
    for b in (SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH):
        try:
            with sdpa_kernel(b):
                F.scaled_dot_product_attention(q.to(dt), q.to(dt), q.to(dt), is_causal=True)
            ok = "能"
        except RuntimeError:
            ok = "不能"
        row += f"{b.name:>22} {ok:<3}"
    log(row)
(RESULTS / "env.txt").write_text("\n".join(lines) + "\n")

## 下载权重

用书里的`download_model.sh`，SmolLM2-135M约272MB，Qwen3-0.6B-Base约1.5GB。

In [ ]:
sh("./download_model.sh", cwd=CODE)
if RUN_QWEN:
    sh("REPO=Qwen/Qwen3-0.6B-Base ./download_model.sh", cwd=CODE)

In [ ]:
def run(name, *args):
    """在code/下跑book/ch14/里的一个脚本，输出边跑边打印，同时存进results/name.txt。"""
    cmd = [sys.executable, "-u", f"book/ch14/{args[0]}", *args[1:]]
    head = f"$ python {' '.join(cmd[2:])}"
    print(head, flush=True)
    out = [head]
    proc = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
        out.append(line.rstrip("\n"))
    proc.wait()
    (RESULTS / f"{name}.txt").write_text("\n".join(out) + "\n")
    if proc.returncode:
        print(f"\n!! 退出码{proc.returncode}，接着跑下一个")

## 第一次核对、半精度：`check_device.py`

基准是同一台机器CPU上的FP32。正文里SmolLM2贪婪生成100个，Qwen3-0.6B-Base生成40个。

In [ ]:
run("check_device_smollm2", "check_device.py", "--device", "cuda")
if RUN_QWEN:
    run("check_device_qwen3", "check_device.py", "--device", "cuda", "--model", "Qwen3-0.6B-Base", "--n", "40")

## 速度：`speed.py`

128个词元，prefill一次喂完，decode每次喂1个。第一行CPU是Colab的虚拟CPU（通常只有2个核），
和作者Mac上的45.0ms不能直接比，正文要比的是同一张表里的CUDA各行。

In [ ]:
run("speed_smollm2", "speed.py", "--device", "cuda")
if RUN_QWEN:
    run("speed_qwen3", "speed.py", "--device", "cuda", "--model", "Qwen3-0.6B-Base")

## 批处理：`batch.py`

先验正确性（4条贪婪序列和单独生成的一字不差），再量B从1到64的吞吐量。

In [ ]:
run("batch_float32", "batch.py", "--device", "cuda", "--dtype", "float32", "--max-batch", str(MAX_BATCH))
run(f"batch_{HALF}", "batch.py", "--device", "cuda", "--dtype", HALF, "--max-batch", str(MAX_BATCH))

## FlashAttention：`flash.py`

一层注意力，手写的那一套和`F.scaled_dot_product_attention()`比耗时和峰值显存，T从512翻倍到16384。
手写的那一套在T=16384时光分数表就要约9.7GB（FP32），T4上多半放不下，脚本会打印“放不下”。

In [ ]:
run("flash_float32", "flash.py", "--device", "cuda", "--dtype", "float32", "--max-t", str(MAX_T))
run(f"flash_{HALF}", "flash.py", "--device", "cuda", "--dtype", HALF, "--max-t", str(MAX_T))

## 打包下载

`results/`里每个脚本一个txt，再加`env.txt`。下载下来放到本机，把各表补进正文。

In [ ]:
tag = torch.cuda.get_device_name(0).replace(" ", "_")
archive = shutil.make_archive(f"/content/ch14_cuda_{tag}", "zip", RESULTS)
print(archive)
for f in sorted(RESULTS.iterdir()):
    print(f"  {f.name}")
from google.colab import files
files.download(archive)